In [2]:
import keras                                                    # ✅ add this
from keras.models import Sequential, Model 
from keras.layers import Conv2D, Flatten, Dense, MaxPooling2D
from keras.constraints import MaxNorm
from keras import activations
import tensorflow as tf
from sklearn.model_selection import train_test_split

import pickle
import logging
import warnings
import numpy as np
import os
import shutil
import statistics as statis

In [30]:

def build_model(nst, nt, nc):
    model = Sequential()

    # Conv and Max pooling1
    model.add(Conv2D(12,(1,3),activation='relu', input_shape=(nst,nt,nc)))
    model.add(MaxPooling2D((1,2)))

    #Conv and Max Pooling2
    model.add(Conv2D(24,(1,3), activation='relu', padding='same'))
    model.add(Conv2D(32,(1,3), activation='relu', padding='same'))
    model.add(MaxPooling2D((1,2)))

    #Conv and Max Pooling3
    model.add(Conv2D(64,(1,3), activation='relu', padding='same'))
    model.add(Conv2D(128,(1,3), activation='relu', padding='same'))
    model.add(MaxPooling2D((1,2)))

    #Conv
    model.add(Conv2D(256,(1,3), activation='relu', padding='same'))

      # Flatten ____________________________________________________
    model.add(Flatten())
    
     # 128 Neurons ____________________________________________________
    model.add(Dense(128, activation='relu',kernel_initializer="normal",kernel_constraint=MaxNorm(3)))    
     # 32 Neurons ____________________________________________________
    model.add(Dense(32, activation='relu',kernel_initializer="normal",kernel_constraint=MaxNorm(3)))
     # 1 Neurons ____________________________________________________
    model.add(Dense(1))

    return model


In [4]:
# Prediction & Errors **************************************************************
def predict(x_test,y_test,model,dirout,dir_case):
    pred = model.predict(x_test)
    y_pred = pred.reshape(len(y_test),)
    yr_pred = np.array([round(valy,1) for valy in y_pred])#round to 1 decimal

    # Statistics value prediction
    pred_error = np.array([round(valerr,1) for valerr in (yr_pred - y_test)])#round to 1 decimal
    abs_pred_error = np.array([round(valea,1) for valea in np.abs(yr_pred - y_test)])#round to 1 decimal
    mean_error = np.mean(abs_pred_error)
    min_error = np.min(abs_pred_error)
    max_error = np.max(abs_pred_error)
    std_error = np.std(pred_error)
    mode_error = statis.mode(abs_pred_error)
    rms_error = np.sqrt(((pred_error)**2).mean())
    
    logging.info('\nPrediction report:')
    logging.info('Mean Absolute Error: %10.3f', round(mean_error,3))
    logging.info('Min Absolute Error: %10.3f', min_error)
    logging.info('Max Absolute Error: %10.3f', max_error)
    logging.info('Std Dev: %10.3f', round(std_error,3))
    logging.info('Mode Absolute Error: %10.3f', mode_error)
    logging.info('RMSE: %10.3f', round(rms_error,3))

    # Save files with prediction results       
    if os.path.exists(dirout+dir_case):
        logging.warning('The folder '+dirout+dir_case+' already exist! It will be removed and made again.\n')
        shutil.rmtree(dirout+dir_case)
    os.mkdir(dirout+dir_case)
    
    f=open(dirout+dir_case+'/Results_Magnitude.dat','w')
    f.write('Magnitude, Predicted Mag\n')
    for ix in range(len(yr_pred)):
        f.write(str(y_test[ix])+' '+ str(yr_pred[ix])+'\n')
    f.close()

    np.savetxt(dirout+dir_case+'/Predict_Eval.txt',(mean_error,min_error,max_error,std_error,
                                                           mode_error,rms_error),
               fmt='%10.5f',header='mean_error, min_error, max_error, std_error, mode_error, rms_error',
               delimiter=' ')

In [ ]:
# Tracking learning rate
def get_lr_metric(optimizer):
    def lr(y_true,y_esti):
        return optimizer.learning_rate
    return lr

# Call checkpoint - Early stopping ****************
def trainCallback(checkpoint_filepath):
    
    model_checkpoint_callback = keras.callbacks.ModelCheckpoint(
        filepath=checkpoint_filepath,
        save_weights_only=True,
        monitor='val_loss',
        mode='min',
        save_best_only=True)

    
    return model_checkpoint_callback


def trainer(nst,nt,nc,dirout,dir_case,x1,y1,x2,y2):
    tf.random.set_seed(2)
    
    # Paths ****************************************************************
    if os.path.exists(dirout+dir_case+'/'):
        logging.warning('The folder '+dirout+dir_case+' already exist! It will be removed and made again.\n')
        shutil.rmtree(dirout+dir_case+'/')
    os.mkdir(dirout+dir_case+'/')

    checkpoint_filepath = dirout+dir_case+'/cp.weights.h5'
    hist_filepath = dirout+dir_case+'/history.p'
    model_path = dirout+dir_case+'/model.keras'
    
    # Parameters ****************************************************************
    batch_size = 128
    epochs = 200
    
    logging.info('Total epochs = %i',epochs)
    logging.info('Batch size = %i', batch_size)
 
    # Build Model ****************************************************************
    model=build_model(nst,nt,nc)
    
    # Learning rate & Optimizer ****************************************************************
    logging.info("\n Using 'standard' learning rate decay...")
    initial_learning_rate = 1e-2
    logging.info('initial learning rate: %5.4f', initial_learning_rate)

    """lr_schedule = keras.optimizers.schedules.ExponentialDecay(
        initial_learning_rate,
        decay_steps=1000,
        decay_rate=0.9,
        staircase=True)"""
        
    steps_per_epoch = len(x1) // batch_size 
    total_steps = steps_per_epoch * epochs   

    lr_schedule = keras.optimizers.schedules.ExponentialDecay(
        initial_learning_rate,
        decay_steps=1000,  # decay ~10 times during training
        decay_rate=0.9,
        staircase=True)

    opt = keras.optimizers.Adam(learning_rate=lr_schedule)
    lr_metric = get_lr_metric(opt)

    # Compile ****************************************************************
    model.compile(loss='mse', optimizer=opt, metrics=['mae',lr_metric])
    
    #Checkpoint ****************************************************************
    model_checkpoint_callback=trainCallback(checkpoint_filepath)
    
    # Train ****************************************************************
    hist = model.fit(x1, y1,validation_data=(x2, y2),epochs=epochs,
                       batch_size=batch_size,verbose=2,
                       callbacks=[model_checkpoint_callback],
                       shuffle=True)
          
    # Evaluate ****************************************************************
    loss, mae, lr = model.evaluate(x2, y2, verbose=2)
    logging.info('\nValidation:')
    logging.info('loss: %5.5f', loss)
    logging.info('mae: %5.5f', mae)
    logging.info('lr: %5.5f', lr)
 
    # Save validation info in file    
    np.savetxt(dirout+dir_case+'/Validation_values.txt',(loss,mae,lr),
           fmt='%5.5f',header='loss, mae, lr',
           delimiter=' ')
    
    # Save Model ****************************************************************
    model.load_weights(checkpoint_filepath)
    model.save(model_path)

    hist_h = hist.history
    
    logging.info('History attributes:')
    logging.info(hist_h.keys())

    with open(hist_filepath, 'wb') as file_pi:
        pickle.dump(hist_h, file_pi)
        
    # Model summary    ****************************************************************
    with open(dirout+dir_case+'/report_model.txt','w', encoding='utf-8') as fh:
        model.summary(print_fn=lambda x: fh.write(x + '\n'))
        
    dot_img_file = dirout+dir_case+'/model.pdf'
    keras.utils.plot_model(model, to_file=dot_img_file, show_shapes=True)
    
    return model

In [6]:
def make_outfolders(dir_out,dir_log,dir_info,dir_trmodel,dir_pred):
    if not os.path.exists(dir_out):
        os.mkdir(dir_out)

    if not os.path.exists(dir_log):
        os.mkdir(dir_log)
        
    if not os.path.exists(dir_info):
        os.mkdir(dir_info)
        
    if not os.path.exists(dir_trmodel):
        os.mkdir(dir_trmodel)
    
    if not os.path.exists(dir_pred):
        os.mkdir(dir_pred)


# Load and split Data *********************************************************
def make_data(dir_info,dir_case):
    dirdata = '../data/processed/tensors/'+dir_case+'/'
    dirout = dir_info+dir_case+'/'
    
    x=np.load(dirdata+'xdata.npy')
    y=np.load(dirdata+'ydata.npy')

    # Split (1): Train-Val & Test Data *********************************************************
    ntest=0.1 # 10%
    ix=np.arange(0, len(y), 1, dtype=int) #index
    x_train1, x_test, ix_train1, ix_test = train_test_split(x, ix,
                                                            test_size=ntest,
                                                            random_state=1)# random_state for reproducible output
    y_train1=y[ix_train1]
    y_test=y[ix_test]
    
    # Split (2): Train and Validation data
    n_val=0.2 # 2% of 90% (1/5 of data train)
    x_train1 = x_train1.astype('float32')
    x_train, x_val, ix_train, ix_val = train_test_split(x_train1, ix_train1,
                                                        test_size=n_val,
                                                        random_state=1)# random_state for reproducible output
    y_train=y[ix_train]
    y_val=y[ix_val]
    
    # Save index in numpy files
    if not os.path.exists(dirout):
        os.mkdir(dirout)
    np.save(dirout+'index_datatrain.npy',ix_train)
    np.save(dirout+'index_dataval.npy',ix_val)
    np.save(dirout+'index_datatest.npy',ix_test)
    
    # Print shapes *********************************************************
    logging.info('Train & Validation data '+str(100-(100*ntest))+'%:')
    logging.info('x_train: '+str(x_train.shape))
    logging.info('y_train: '+str(y_train.shape))
    logging.info('x_val: '+str(x_val.shape))
    logging.info('y_val: '+str(y_val.shape))
    logging.info('Test_data'+str(100*ntest)+'%:')
    logging.info('x_test: '+str(x_test.shape))
    logging.info('y_test: '+str(y_test.shape)+'\n')

    return (x_train,y_train), (x_val,y_val), (x_test,y_test)




In [29]:
# dataset shape --> Set dimension!!!
nt = 181 # Time windows [seconds] #Set 181 or 501
nst = 3 # Nunmber of stations #Set 3 or 7
nc = 3 # Channels. Don't change it.

case_nm = 'GNSS_M'+str(nst)+'S_'+str(nt)

# Paths # Set the folder and file names!
project_root = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
dir_out = os.path.join(project_root, 'results')
dir_datinf = dir_out+'/data_info_1/'
dir_trmodel = dir_out+'/models_1/'
dir_pred = dir_out+'/predictions_1/'
dir_log = dir_out+'/out_log/'
filelog = case_nm +'.log'



# ****************************************************************************
def main():
    
    # Create the output folders
    make_outfolders(dir_out,dir_log,dir_datinf,dir_trmodel,dir_pred)
        
    logging.basicConfig(
        filename=dir_log+filelog,
        level=logging.DEBUG,
        format='%(asctime)s [%(levelname)s] %(name)s: %(message)s',
        datefmt='%H:%M:%S'
    )

    # Make datasets
    (x_train,y_train), (x_val,y_val), (x_test,y_test) = make_data(dir_datinf,case_nm)
    
    # Training Model
    model = trainer(nst,nt,nc,dir_trmodel,case_nm,x_train,y_train,x_val,y_val)
    
    # Predictions
    predict(x_test,y_test,model,dir_pred,case_nm)
    
    logging.info('Finished *********************************************************************')
    
if __name__ == '__main__':

    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", category=FutureWarning)
        
        main()


C:\Users\hp\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/200
12/12 - 4s - 322ms/step - loss: 28.0379 - lr: 0.0010 - mae: 4.6925 - val_loss: 4.1843 - val_lr: 0.0010 - val_mae: 1.9562
Epoch 2/200
12/12 - 1s - 52ms/step - loss: 4.0898 - lr: 0.0010 - mae: 1.8181 - val_loss: 0.5597 - val_lr: 0.0010 - val_mae: 0.5959
Epoch 3/200
12/12 - 1s - 44ms/step - loss: 1.1542 - lr: 0.0010 - mae: 0.8957 - val_loss: 0.9544 - val_lr: 0.0010 - val_mae: 0.8024
Epoch 4/200
12/12 - 1s - 51ms/step - loss: 0.5796 - lr: 0.0010 - mae: 0.6147 - val_loss: 0.5186 - val_lr: 0.0010 - val_mae: 0.5735
Epoch 5/200
12/12 - 1s - 51ms/step - loss: 0.3931 - lr: 0.0010 - mae: 0.5055 - val_loss: 0.3819 - val_lr: 0.0010 - val_mae: 0.5169
Epoch 6/200
12/12 - 1s - 65ms/step - loss: 0.3327 - lr: 0.0010 - mae: 0.4721 - val_loss: 0.3611 - val_lr: 0.0010 - val_mae: 0.5059
Epoch 7/200
12/12 - 1s - 55ms/step - loss: 0.3167 - lr: 0.0010 - mae: 0.4636 - val_loss: 0.3627 - val_lr: 0.0010 - val_mae: 0.5079
Epoch 8/200
12/12 - 1s - 55ms/step - loss: 0.3106 - lr: 0.0010 - mae: 0.4610 - va

You must install graphviz (see instructions at https://graphviz.gitlab.io/download/) for `plot_model` to work.
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 281ms/step


In [22]:
import numpy as np

x = np.load('../data/processed/tensors/GNSS_M3S_181/xdata.npy')
y = np.load('../data/processed/tensors/GNSS_M3S_181/ydata.npy')

print("=== SHAPE ===")
print(f"x : {x.shape}")   # (n_samples, nst, nt, nc) ?
print(f"y : {y.shape}")

print("\n=== y (magnitudes) ===")
print(f"mean   : {np.mean(y):.4f}")
print(f"std    : {np.std(y):.4f}")
print(f"min    : {np.min(y):.4f}")
print(f"max    : {np.max(y):.4f}")

print("\n=== x (signal GNSS) ===")
print(f"mean   : {np.mean(x):.4f}")
print(f"std    : {np.std(x):.4f}")
print(f"min    : {np.min(x):.4f}")
print(f"max    : {np.max(x):.4f}")
print(f"NaN    : {np.sum(np.isnan(x))}")
print(f"Inf    : {np.sum(np.isinf(x))}")

print("\n=== variance par échantillon ===")
per_std = np.std(x, axis=(1,2,3))
print(f"std moyenne : {per_std.mean():.4f}")
print(f"std min     : {per_std.min():.4f}")
print(f"std max     : {per_std.max():.4f}")
print(f"échantillons plats (std<0.001) : {np.sum(per_std < 0.001)}")

=== SHAPE ===
x : (500, 3, 181, 3)
y : (500,)

=== y (magnitudes) ===
mean   : 7.3938
std    : 0.4571
min    : 6.6000
max    : 8.2000

=== x (signal GNSS) ===
mean   : 0.0000
std    : 0.0772
min    : -0.3790
max    : 0.3915
NaN    : 0
Inf    : 0

=== variance par échantillon ===
std moyenne : 0.0768
std min     : 0.0517
std max     : 0.0999
échantillons plats (std<0.001) : 0


In [23]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle
import os

def visualize_model_results(dirout, dir_case):
    results_dir = os.path.join(dirout, dir_case)
    history_file = os.path.join(results_dir, 'history.p')
    magnitude_file = os.path.join(results_dir, 'Results_Magnitude.dat')
    
    # Create a directory to save the plots
    plots_dir = os.path.join(results_dir, 'plots')
    if not os.path.exists(plots_dir):
        os.mkdir(plots_dir)

    # ==========================================
    # 1. Plot Training History (Overfitting Check)
    # ==========================================
    if os.path.exists(history_file):
        with open(history_file, 'rb') as f:
            hist = pickle.load(f)
            
        epochs = range(1, len(hist['loss']) + 1)
        
        # Plot Loss (MSE)
        plt.figure(figsize=(10, 6))
        plt.plot(epochs, hist['loss'], 'b-', label='Training Loss (MSE)')
        plt.plot(epochs, hist['val_loss'], 'r-', label='Validation Loss (MSE)')
        plt.title('Learning Curves: Check for Overfitting')
        plt.xlabel('Epochs')
        plt.ylabel('Loss (Mean Squared Error)')
        plt.legend()
        plt.grid(True, linestyle='--', alpha=0.7)
        plt.savefig(os.path.join(plots_dir, 'learning_curve_loss.png'), bbox_inches='tight')
        plt.close()

        # Plot MAE
        plt.figure(figsize=(10, 6))
        plt.plot(epochs, hist['mae'], 'b-', label='Training MAE')
        plt.plot(epochs, hist['val_mae'], 'r-', label='Validation MAE')
        plt.title('Mean Absolute Error Progression')
        plt.xlabel('Epochs')
        plt.ylabel('MAE (Magnitude)')
        plt.legend()
        plt.grid(True, linestyle='--', alpha=0.7)
        plt.savefig(os.path.join(plots_dir, 'learning_curve_mae.png'), bbox_inches='tight')
        plt.close()

        # Plot Learning Rate
        if 'lr' in hist:
            plt.figure(figsize=(10, 6))
            plt.plot(epochs, hist['lr'], 'g-', label='Learning Rate')
            plt.title('Learning Rate Decay (Staircase)')
            plt.xlabel('Epochs')
            plt.ylabel('Learning Rate')
            plt.legend()
            plt.grid(True, linestyle='--', alpha=0.7)
            plt.savefig(os.path.join(plots_dir, 'learning_rate_decay.png'), bbox_inches='tight')
            plt.close()
            print("History plots saved successfully.")
    else:
        print(f"Warning: {history_file} not found.")

    # ==========================================
    # 2. Plot Prediction Performance
    # ==========================================
    if os.path.exists(magnitude_file):
        # Read the space-separated dat file (skipping the header)
        data = pd.read_csv(magnitude_file, sep='\s+', skiprows=1, names=['Actual', 'Predicted'])
        
        actual_mag = data['Actual']
        predicted_mag = data['Predicted']
        errors = predicted_mag - actual_mag
        
        # Parity Plot (Actual vs Predicted)
        plt.figure(figsize=(8, 8))
        plt.scatter(actual_mag, predicted_mag, alpha=0.5, c='blue', edgecolors='k')
        
        # Draw the y = x reference line
        min_val = min(actual_mag.min(), predicted_mag.min())
        max_val = max(actual_mag.max(), predicted_mag.max())
        plt.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Ideal Prediction (y=x)')
        
        plt.title('Parity Plot: Actual vs Predicted Magnitudes')
        plt.xlabel('Actual Magnitude')
        plt.ylabel('Predicted Magnitude')
        plt.legend()
        plt.grid(True, linestyle='--', alpha=0.7)
        plt.axis('equal') # Ensures the scale is exactly the same on both axes
        plt.savefig(os.path.join(plots_dir, 'parity_plot.png'), bbox_inches='tight')
        plt.close()

        # Error Distribution Histogram
        plt.figure(figsize=(10, 6))
        plt.hist(errors, bins=30, color='orange', edgecolor='black', alpha=0.7)
        plt.axvline(x=0, color='red', linestyle='--', linewidth=2, label='Zero Error')
        plt.title('Error Distribution (Predicted - Actual)')
        plt.xlabel('Error in Magnitude')
        plt.ylabel('Frequency')
        plt.legend()
        plt.grid(True, axis='y', linestyle='--', alpha=0.7)
        plt.savefig(os.path.join(plots_dir, 'error_histogram.png'), bbox_inches='tight')
        plt.close()
        print("Prediction plots saved successfully.")
    else:
        print(f"Warning: {magnitude_file} not found.")

# --- Execution Example ---
# Replace with your actual directories used in utils.make_outfolders
# visualize_model_results(dirout='./results/', dir_case='case_1')

In [27]:
import numpy as np
x = np.load('../data/processed/tensors/GNSS_M7S_181/xdata.npy')
y = np.load('../data/processed/tensors/GNSS_M7S_181/ydata.npy')
print("Total samples:", x.shape[0])
print("Total samples:", y.shape[0])  

Total samples: 34567
Total samples: 34567


In [9]:
import modal
import os

# ── 1. Modal image with all dependencies ──────────────────────────────────────
image = (
    modal.Image.debian_slim(python_version="3.11")
    .pip_install(
        "tensorflow[and-cuda]",
        "keras",
        "scikit-learn",
        "numpy",
        "pydot",
        "graphviz",
    )
    .apt_install("graphviz")
)

# ── 2. Persistent volume for BOTH data and results ────────────────────────────
volume = modal.Volume.from_name("earthquake-dl-vol", create_if_missing=True)
VOLUME_PATH = "/vol"

app = modal.App("earthquake-dl-hrgnss", image=image)


# ── 3. Training function (runs on Modal GPU) ───────────────────────────────────
@app.function(
    gpu="A10G",
    timeout=60 * 60 * 3,
    volumes={VOLUME_PATH: volume},
)
def train(nst: int = 7, nt: int = 501, nc: int = 3):
    import keras
    import tensorflow as tf
    import numpy as np
    import pickle
    import logging

    from sklearn.model_selection import train_test_split

    # ── Config ────────────────────────────────────────────────────────────────
    case_nm     = f"GNSS_M{nst}S_{nt}"
    data_dir    = f"{VOLUME_PATH}/data/{case_nm}/"
    dir_out     = f"{VOLUME_PATH}/results"
    dir_log     = f"{dir_out}/out_log/"
    dir_datinf  = f"{dir_out}/data_info_1/{case_nm}/"
    dir_trmodel = f"{dir_out}/models_1/{case_nm}/"
    dir_pred    = f"{dir_out}/predictions_1/{case_nm}/"

    for d in [dir_log, dir_datinf, dir_trmodel, dir_pred]:
        os.makedirs(d, exist_ok=True)

    logging.basicConfig(
        filename=f"{dir_log}{case_nm}.log",
        level=logging.DEBUG,
        format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
        datefmt="%H:%M:%S",
    )

    # ── Model ─────────────────────────────────────────────────────────────────
    def build_model(nst, nt, nc):
        from keras.models import Sequential
        from keras.layers import Conv2D, Flatten, Dense, MaxPooling2D
        from keras.constraints import MaxNorm

        model = Sequential([
            Conv2D(12,  (1,3), activation="relu", input_shape=(nst, nt, nc)),
            MaxPooling2D((1,2)),
            Conv2D(24,  (1,3), activation="relu", padding="same"),
            Conv2D(32,  (1,3), activation="relu", padding="same"),
            MaxPooling2D((1,2)),
            Conv2D(64,  (1,3), activation="relu", padding="same"),
            Conv2D(128, (1,3), activation="relu", padding="same"),
            MaxPooling2D((1,2)),
            Conv2D(256, (1,3), activation="relu", padding="same"),
            Flatten(),
            Dense(128, activation="relu", kernel_initializer="normal", kernel_constraint=MaxNorm(3)),
            Dense(32,  activation="relu", kernel_initializer="normal", kernel_constraint=MaxNorm(3)),
            Dense(1),
        ])
        return model

    def get_lr_metric(optimizer):
        def lr(y_true, y_esti):
            return optimizer.learning_rate
        return lr

    # ── Load data from volume ─────────────────────────────────────────────────
    print(f"Loading data from {data_dir}")
    x = np.load(data_dir + "xdata.npy")
    y = np.load(data_dir + "ydata.npy")
    print(f"Dataset shape: x={x.shape}, y={y.shape}")

    ix = np.arange(len(y), dtype=int)
    x_tr1, x_test, ix_tr1, ix_test = train_test_split(x, ix, test_size=0.1,  random_state=1)
    x_train, x_val, ix_train, ix_val = train_test_split(x_tr1, ix_tr1, test_size=0.2, random_state=1)
    y_train, y_val, y_test = y[ix_train], y[ix_val], y[ix_test]

    np.save(dir_datinf + "index_datatrain.npy", ix_train)
    np.save(dir_datinf + "index_dataval.npy",   ix_val)
    np.save(dir_datinf + "index_datatest.npy",  ix_test)

    logging.info("x_train=%s  x_val=%s  x_test=%s", x_train.shape, x_val.shape, x_test.shape)

    # ── Train ─────────────────────────────────────────────────────────────────
    tf.random.set_seed(2)

    batch_size      = 128
    epochs          = 200
    steps_per_epoch = max(1, len(x_train) // batch_size)

    lr_schedule = keras.optimizers.schedules.ExponentialDecay(
        initial_learning_rate=1e-2,
        decay_steps=steps_per_epoch,
        decay_rate=0.9,
        staircase=True,
    )

    opt       = keras.optimizers.Adam(learning_rate=lr_schedule)
    lr_metric = get_lr_metric(opt)
    model     = build_model(nst, nt, nc)
    model.compile(loss="mse", optimizer=opt, metrics=["mae", lr_metric])

    checkpoint_cb = keras.callbacks.ModelCheckpoint(
        filepath=dir_trmodel + "cp.weights.h5",
        save_weights_only=True,
        monitor="val_loss",
        mode="min",
        save_best_only=True,
    )

    model.fit(
        x_train, y_train,
        validation_data=(x_val, y_val),
        epochs=epochs,
        batch_size=batch_size,
        verbose=2,
        callbacks=[checkpoint_cb],
        shuffle=True,
    )

    # ── Evaluate ──────────────────────────────────────────────────────────────
    loss, mae, lr = model.evaluate(x_val, y_val, verbose=2)
    logging.info("loss=%.5f  mae=%.5f  lr=%.5f", loss, mae, lr)
    np.savetxt(dir_trmodel + "Validation_values.txt", (loss, mae, lr),
               fmt="%5.5f", header="loss, mae, lr")

    model.load_weights(dir_trmodel + "cp.weights.h5")
    model.save(dir_trmodel + "model.keras")

    with open(dir_trmodel + "history.p", "wb") as f:
        pickle.dump(model.history.history, f)

    with open(dir_trmodel + "report_model.txt", "w") as fh:
        model.summary(print_fn=lambda x: fh.write(x + "\n"))

    # ── Predict ───────────────────────────────────────────────────────────────
    y_pred      = model.predict(x_test).reshape(len(y_test),)
    yr_pred     = np.array([round(v, 1) for v in y_pred])
    pred_error  = yr_pred - y_test
    abs_error   = np.abs(pred_error)

    with open(dir_pred + "Results_Magnitude.dat", "w") as f:
        f.write("Magnitude, Predicted Mag\n")
        for i in range(len(yr_pred)):
            f.write(f"{y_test[i]} {yr_pred[i]}\n")

    np.savetxt(dir_pred + "Predict_Eval.txt",
               (np.mean(abs_error), np.min(abs_error), np.max(abs_error),
                np.std(pred_error), np.sqrt((pred_error**2).mean())),
               fmt="%10.5f",
               header="mean_error, min_error, max_error, std_error, rms_error")

    volume.commit()  # persist everything to volume

    print(f"\n✅ Done!  loss={loss:.5f}  mae={mae:.5f}  lr={lr:.5f}")
    return {"loss": float(loss), "mae": float(mae), "lr": float(lr)}


# ── 4. Upload local data + run training + download results ────────────────────
@app.local_entrypoint()
def main():
    import pathlib

    nst, nt  = 7, 501                          # ← change here if needed
    case_nm  = f"GNSS_M{nst}S_{nt}"
    local_data = pathlib.Path(f"data/processed/tensors/{case_nm}")

    # Step 1: upload .npy files to volume
    print(f"Uploading {case_nm} data to Modal volume...")
    for npy_file in local_data.glob("*.npy"):
        remote_path = f"/data/{case_nm}/{npy_file.name}"
        with npy_file.open("rb") as f:
            volume.put_file(f, remote_path)
        print(f"  ✅ Uploaded {npy_file.name}")
    volume.commit()
    print("Upload complete.\n")

    # Step 2: run training on GPU
    result = train.remote(nst=nst, nt=nt, nc=3)
    print("\nFinal metrics:", result)

    # Step 3: download results back to local ./results/
    print("\nDownloading results...")
    for entry in volume.listdir("/results", recursive=True):
        local_path = pathlib.Path(".") / entry.path.lstrip("/")
        local_path.parent.mkdir(parents=True, exist_ok=True)
        if not entry.is_dir:
            with local_path.open("wb") as f:
                for chunk in volume.read_file(entry.path):
                    f.write(chunk)
    print("✅ Results saved to ./results/")